# 第6章 分类模型 · 限时测试



本册为依据课程材料新编的训练题，不是教师公布的考试原题。

## 材料依据
- [[课程/深度学习/PPT/Chapter6/Chapter6.md]]
- [[课程/深度学习/演示代码/Chapter6.py]]
- [[课程/深度学习/课堂笔记/2026-10-12-第11节-深度学习编程与分类模型宣讲.md]]
- [[课程/深度学习/课堂笔记/2026-10-14-第12节-神经网络分类模型代码编写.md]]
- [[课程/深度学习/课堂笔记/2026-10-19-第13节-神经网络与卷积网络课程宣讲.md]]（仅分类部分，不含卷积/池化）
- [[课程/深度学习/作业/Homework03/Homework03.md]]
- [[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md]]


## 模拟要求

计时 **30 分钟，100 分**；一道完整编程题、多个小问，**从导入库开始独立编写，不做代码补全**。先作答，再运行自检。题面中的接口名称仅用于统一自检，不提供实现框架。

本地 PNG 写入当前目录 `output/`（请用 pathlib 创建目录）；正式考试若题目指定 `/output/Qxxx.png`，须严格使用正式路径和文件名。练习中的输出目录是本地适配，不是正式考试环境的改动。


## Q6B 30分钟综合题：同心三区识别与稳定损失

**新编迁移题，依据** [[课程/深度学习/PPT/Chapter6/Chapter6.md]]“Softmax结合交叉熵的计算”小节、[[课程/深度学习/课堂笔记/2026-10-19-第13节-神经网络与卷积网络课程宣讲.md#三、从概率矩阵得到准确率与动画（核心）]]；不同于A卷的符号四区，类别边界由半径决定。

**共100分，30分钟，从空白独立编程。**

1. **数据（20分）**：导入库，CPU单线程、torch.manual_seed(62)。均匀[-1.5,1.5]^2顺序生成600训练/200测试点Xb6_train/Xb6_test。实现`ring_labels_b6(x)`：r²<.5类0；.5≤r²<1.5类1；r²≥1.5类2，返回long标签yb6_train/yb6_test，边界归外圈。
2. **网络与稳定损失（20分）**：net_b6为2→24→24→3、ReLU隐藏、raw logits输出。实现`stable_ce_b6(z,y)`，逐行减max后计算log(sum(exp(shifted)))−shifted_true并mean，返回可反传标量；用`[[1000,999,-1000]]`、真实类0与nn.CrossEntropyLoss对照。
3. **训练（30分）**：Adam(.02)，140步全批，用自己实现的stable_ce_b6；每步新前向并清梯度，记录history_b6。不得先Softmax再log导致极端数值失稳。
4. **预测（15分）**：no_grad输出pred_b6=logits.argmax(1)、acc_b6、最终train_ce_b6；标签已经0起始，不能加一。
5. **图与解释（15分）**：保存output/Q6B.png，两个同色映射散点图显示测试True classes/Predicted classes，英文标题/轴名，相同范围；可加训练曲线作第三图。解释为何sigmoid标量输出不能直接替代本题三类logits+CE，以及为何低CE与高accuracy不完全等价。

评分按上述功能，不能以恰好某个准确率作为全部评分；不要求庞大调参。


In [ ]:
# 从导入库开始，在此独立完成整道编程题。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
import numpy as np
import torch
from pathlib import Path
try:
    assert ring_labels_b6(torch.tensor([[0.,0.],[.5,.5],[1.,1.],[1.,0.]])).tolist()==[0,1,2,1]
    assert ring_labels_b6(torch.tensor([[0.,1.25]])).item()==2
    extreme_b6 = torch.tensor([[1000.,999.,-1000.],[-1000.,1000.,0.]],dtype=torch.float64,requires_grad=True)
    target_b6 = torch.tensor([0,0])
    manual_b6 = stable_ce_b6(extreme_b6,target_b6)
    assert torch.isfinite(manual_b6) and torch.allclose(manual_b6,nn.CrossEntropyLoss()(extreme_b6,target_b6))
    manual_b6.backward()
    assert torch.isfinite(extreme_b6.grad).all()
    assert net_b6(torch.zeros(1,2)).shape==(1,3)
    assert sum(p.numel() for p in net_b6.parameters())==747
    assert len(history_b6)==140 and np.isfinite(history_b6).all()
    assert 0<=pred_b6.min()<=pred_b6.max()<=2 and .35<=acc_b6<=1
    assert Path("output/Q6B.png").stat().st_size>1000
except (NameError, NotImplementedError) as exc:
    print("尚未完成本题：", exc)
else:
    print("自检通过；仍需人工核对题目全部要求。")
